<a href="https://colab.research.google.com/github/dhag/colab_test/blob/main/AI_04_A_%E5%AE%9F%E7%BF%92A_%E3%83%86%E3%82%A3%E3%83%BC%E3%83%81%E3%83%A3%E3%83%96%E3%83%AB%E3%83%9E%E3%82%B7%E3%83%B3%E3%81%A7%E4%BD%9C%E3%81%A3%E3%81%9F%E3%83%A2%E3%83%87%E3%83%AB%E3%81%A7%E7%94%BB%E5%83%8F%E8%AA%8D%E8%AD%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Teachable Machine のモデルで画像認識

Google Colab上で、Teachable Machineから出力した `keras_Model.h5` と `labels.txt` を使って画像認識します。

※ `.h5` は元ファイルを直接変更せず、一時的な修正版を作って読み込みます。


In [1]:
!pip install -q tf-keras

In [2]:
import os
import io
import json
import shutil
import tempfile
import re

import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
import tf_keras
import h5py

print("TensorFlow:", tf.__version__)
print("tf_keras:", tf_keras.__version__)

TensorFlow: 2.21.0
tf_keras: 2.21.0


## 1. Teachable Machine のモデルとラベルをアップロード

`keras_Model.h5` と `labels.txt` を順番に選択してください。


In [4]:
from google.colab import files

print('【1/2】Teachable Machine の keras_Model.h5 を選択してください')
uploaded_model = files.upload()
if not uploaded_model:
    raise RuntimeError('モデルファイルがアップロードされませんでした。')
model_path = next(iter(uploaded_model))

print('\n【2/2】Teachable Machine の labels.txt を選択してください')
uploaded_labels = files.upload()
if not uploaded_labels:
    raise RuntimeError('labels.txt がアップロードされませんでした。')
labels_path = next(iter(uploaded_labels))

print(f'\nモデル: {model_path}')
print(f'ラベル: {labels_path}')


【1/2】Teachable Machine の keras_Model.h5 を選択してください


Saving keras_model.h5 to keras_model (1).h5

【2/2】Teachable Machine の labels.txt を選択してください


Saving labels.txt to labels (1).txt

モデル: keras_model (1).h5
ラベル: labels (1).txt


## 2. モデルを読み込む

Teachable Machine の古い `.h5` モデルでは、現在の Keras で `DepthwiseConv2D` の `groups` に関する互換性エラーが発生することがあります。
その場合だけ、一時コピーから `groups` を削除して読み込みます。元の `.h5` は変更しません。


In [5]:
def remove_depthwise_groups(obj):
    """
    古いTeachable Machine H5とKerasの互換性対策。
    DepthwiseConv2Dのgroupsだけ削除する。
    """
    if isinstance(obj, dict):
        if obj.get("class_name") == "DepthwiseConv2D":
            config = obj.get("config", {})
            config.pop("groups", None)

        for key, value in obj.items():
            obj[key] = remove_depthwise_groups(value)

    elif isinstance(obj, list):
        obj = [remove_depthwise_groups(x) for x in obj]

    return obj


def make_compatible_h5(src_path):
    """
    元のH5を変更せず、一時コピーを作成する。
    """
    fd, dst_path = tempfile.mkstemp(suffix=".h5")
    os.close(fd)

    shutil.copy2(src_path, dst_path)

    with h5py.File(dst_path, "r+") as f:

        if "model_config" not in f.attrs:
            raise RuntimeError(
                "H5ファイルにmodel_configがありません。"
            )

        model_config = f.attrs["model_config"]

        if isinstance(model_config, bytes):
            model_config = model_config.decode("utf-8")

        config = json.loads(model_config)

        config = remove_depthwise_groups(config)

        f.attrs.modify(
            "model_config",
            json.dumps(config).encode("utf-8")
        )

    return dst_path


def load_teachable_machine_model(path):

    # --------------------------------------------------
    # 1. まず旧Keras互換ローダーでそのまま読む
    # --------------------------------------------------
    try:
        model = tf_keras.models.load_model(
            path,
            compile=False
        )

        print("✓ tf_kerasでモデルを読み込みました")
        return model

    except Exception as e:

        print("通常のtf_keras読み込みに失敗しました")
        print(type(e).__name__, ":", e)

    # --------------------------------------------------
    # 2. DepthwiseConv2D.groups対策
    # --------------------------------------------------
    fixed_path = make_compatible_h5(path)

    try:
        model = tf_keras.models.load_model(
            fixed_path,
            compile=False
        )

        print("✓ 互換性修正版H5をtf_kerasで読み込みました")
        print("✓ 元のH5ファイルは変更していません")

        return model

    except Exception as e:

        raise RuntimeError(
            "\n"
            "Teachable MachineのH5モデルを読み込めませんでした。\n\n"
            f"{type(e).__name__}: {e}\n\n"
            "keras_Model.h5自体の構造を確認する必要があります。"
        ) from e

In [6]:
model = load_teachable_machine_model(model_path)

print("\n入力shape:")
print(model.input_shape)

print("\n出力shape:")
print(model.output_shape)

model.summary()


✓ tf_kerasでモデルを読み込みました

入力shape:
(None, 224, 224, 3)

出力shape:
(None, 2)
Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequential_1 (Sequential)   (None, 1280)              410208    
                                                                 
 sequential_3 (Sequential)   (None, 2)                 128300    
                                                                 
Total params: 538508 (2.05 MB)
Trainable params: 524428 (2.00 MB)
Non-trainable params: 14080 (55.00 KB)
_________________________________________________________________


In [7]:
def load_class_names(path):
    with open(path, 'r', encoding='utf-8') as f:
        labels = [line.strip() for line in f if line.strip()]

    # Teachable Machineの labels.txt は「0 クラス名」の形式の場合がある。
    cleaned = []
    for label in labels:
        m = re.match(r'^\s*\d+\s+(.*)$', label)
        cleaned.append(m.group(1).strip() if m else label)
    return cleaned

class_names = load_class_names(labels_path)
print(f'{len(class_names)} クラス:')
for i, name in enumerate(class_names):
    print(f'  {i}: {name}')


2 クラス:
  0: Class 1
  1: Class 2


## 3. 画像ファイルをアップロードして認識

まずはこちらで動作確認することをおすすめします。JPEG / PNGなどの画像を選択してください。


In [ ]:
def preprocess_image(img_bgr):
    # Teachable Machineの標準的な入力サイズ
    h, w = model.input_shape[1], model.input_shape[2]
    if h is None or w is None:
        h, w = 224, 224

    resized = cv2.resize(img_bgr, (w, h), interpolation=cv2.INTER_AREA)
    x = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB).astype(np.float32)
    x = x / 127.5 - 1.0
    x = np.expand_dims(x, axis=0)
    return resized, x

def predict_image(img_bgr):
    resized, x = preprocess_image(img_bgr)
    prediction = model.predict(x, verbose=0)[0]

    # 出力クラス数とlabels.txtの数が違う場合を明示的に通知
    if len(prediction) != len(class_names):
        raise RuntimeError(
            f'モデルの出力クラス数({len(prediction)})と '
            f'labels.txtのクラス数({len(class_names)})が一致しません。'
        )

    # 念のため確率として扱えるようにする
    if np.any(prediction < 0) or not np.isclose(prediction.sum(), 1.0, atol=1e-3):
        prediction = tf.nn.softmax(prediction).numpy()

    top_indices = np.argsort(prediction)[::-1]
    best = int(top_indices[0])

    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(resized, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title(
        f'予測: {class_names[best]}\n信頼度: {prediction[best] * 100:.2f}%',
        fontsize=14,
        fontweight='bold'
    )
    plt.show()

    print(f'予測: {class_names[best]}')
    print(f'信頼度: {prediction[best] * 100:.2f}%')
    print('\n上位3つの予測:')
    for rank, idx in enumerate(top_indices[:min(3, len(top_indices))], 1):
        print(f'  {rank}. {class_names[idx]}: {prediction[idx] * 100:.2f}%')

print('画像ファイルを選択してください。')
uploaded_image = files.upload()
if uploaded_image:
    image_name = next(iter(uploaded_image))
    image_pil = Image.open(io.BytesIO(uploaded_image[image_name])).convert('RGB')
    image_bgr = cv2.cvtColor(np.array(image_pil), cv2.COLOR_RGB2BGR)
    predict_image(image_bgr)
else:
    print('画像がアップロードされませんでした。')


## 4. カメラで撮影して認識（Google Colab）

上の画像アップロードが正常に動いた後、こちらを実行してください。


In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='captured_image.jpg', quality=0.8):
    js = Javascript(r'''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const button = document.createElement('button');
      button.textContent = '撮影する';
      button.style.padding = '10px 20px';
      button.style.fontSize = '16px';
      button.style.cursor = 'pointer';

      const video = document.createElement('video');
      video.style.display = 'block';
      video.style.maxWidth = '100%';
      video.style.marginTop = '10px';

      const status = document.createElement('p');
      status.textContent = 'カメラへのアクセスを許可してください。';

      div.appendChild(status);
      div.appendChild(button);
      div.appendChild(video);
      document.body.appendChild(div);

      let stream;
      try {
        stream = await navigator.mediaDevices.getUserMedia({video: true, audio: false});
        video.srcObject = stream;
        await video.play();
        status.textContent = 'カメラの映像を確認し、「撮影する」を押してください。';

        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        await new Promise(resolve => button.onclick = resolve);

        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);

        return canvas.toDataURL('image/jpeg', quality);
      } finally {
        if (stream) stream.getTracks().forEach(track => track.stop());
        div.remove();
      }
    }
    ''')
    display(js)
    data = eval_js(f'takePhoto({quality})')
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

try:
    filename = take_photo()
    img = cv2.imread(filename)
    if img is None:
        raise RuntimeError('撮影した画像を読み込めませんでした。')
    predict_image(img)
except Exception as e:
    print('カメラ認識でエラーが発生しました:')
    print(type(e).__name__, ':', e)
    print('\nカメラが使えない場合は、前の「画像ファイルをアップロードして認識」を使ってください。')
